# Why Python loops are slow in JAX

## The problem

In standard Python, this is how you'd compute a running sum:

```python
total = 0
for x in xs:
    total = total + x
```

But in JAX, this is **extremely slow** (10-1000× slower than it should be).

## Why?

JAX operations are designed to be **traced once and compiled**. Every time 
you call a Python `for` loop, JAX sees each iteration as a separate call:
- Dispatch overhead per iteration
- GPU synchronization per iteration  
- No fusion of operations across iterations

For a loop with 1000 iterations, that's 1000 separate kernel launches.

## The solution: `lax.scan`

`lax.scan` tells JAX: "Here is the ENTIRE loop structure. Compile it as one 
fused operation." This lets JAX:
- Fuse operations across iterations
- Minimize GPU memory transfers
- Compile the loop body once and reuse it

## The rule

**Never use a Python `for` loop in performance-critical JAX code.** Use `lax.scan` 
(or `lax.while_loop` for unknown counts) instead.

## When Python loops are OK

- Setup code (loading data, building structures) — runs once
- Debugging — readability matters more than speed
- Loops with data-dependent control flow — needs `lax.while_loop`

In [1]:
import jax
import jax.numpy as jnp
import time

xs = jnp.arange(1000)

# Method 1: Python for loop (slow)
def sum_python(xs):
    total = jnp.array(0.0)
    for x in xs:
        total = total + x
    return total

# Method 2: jnp.sum (fast, but only works for built-in reductions)
def sum_native(xs):
    return jnp.sum(xs)

# Method 3: lax.scan (fast, works for arbitrary loop logic)
def sum_scan(xs):
    def body(carry, x):
        return carry + x, None  # (new_carry, output)
    final, _ = jax.lax.scan(body, jnp.array(0.0), xs)
    return final

# Warm up (JIT compile)
_ = jax.jit(sum_python)(xs).block_until_ready()
_ = jax.jit(sum_native)(xs).block_until_ready()
_ = jax.jit(sum_scan)(xs).block_until_ready()

# Time each method
N = 100

start = time.time()
for _ in range(N):
    sum_python(xs).block_until_ready()
t_python = time.time() - start

start = time.time()
for _ in range(N):
    sum_native(xs).block_until_ready()
t_native = time.time() - start

start = time.time()
for _ in range(N):
    sum_scan(xs).block_until_ready()
t_scan = time.time() - start

print(f"Python for loop:  {t_python*1000:.2f} ms (avg of {N} runs)")
print(f"jnp.sum:          {t_native*1000:.2f} ms (avg of {N} runs)")
print(f"lax.scan:         {t_scan*1000:.2f} ms (avg of {N} runs)")
print()
print(f"Speedup scan vs python: {t_python/t_scan:.0f}x")

Python for loop:  1539.04 ms (avg of 100 runs)
jnp.sum:          22.16 ms (avg of 100 runs)
lax.scan:         2976.48 ms (avg of 100 runs)

Speedup scan vs python: 1x


# lax.scan basics

`lax.scan` is JAX's way to express a `for` loop that compiles efficiently.

## Function signature

```python
jax.lax.scan(f, init, xs)
```

- `f`: The loop body function with signature `(carry, x) -> (new_carry, y)`
- `init`: The initial value of the carry (loop state)
- `xs`: The array to iterate over (can be any leading dimension)

## The body function

The body function receives two arguments:
1. `carry`: The state passed from the previous iteration
2. `x`: The current element from `xs`

It returns two values:
1. `new_carry`: The updated state for the next iteration
2. `y`: An output for this iteration (or `None` if you don't need outputs)

## What scan returns

```python
final_carry, ys = jax.lax.scan(f, init, xs)
```

- `final_carry`: The carry value after the last iteration
- `ys`: A stacked array of all the `y` outputs (shape: `xs.shape + y.shape`)

## Mental model

Think of `scan` as a "for loop on the GPU":
- The body function is the loop body
- `init` is the starting state
- `xs` is what you iterate over
- `final_carry` is the final state
- `ys` is the collected output (like a list of results, but pre-allocated)

In [2]:
# Example 1: Simple cumulative sum
xs = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])

def body(carry, x):
    new_carry = carry + x
    return new_carry, new_carry  # also output the running sum

final, all_running_sums = jax.lax.scan(body, jnp.array(0.0), xs)

print(f"Input:       {xs}")
print(f"Final sum:   {final}")
print(f"All running: {all_running_sums}")
print()

# Example 2: Body with no output
def body_no_output(carry, x):
    return carry + x, None  # discard per-iteration output

final, _ = jax.lax.scan(body_no_output, jnp.array(0.0), xs)
print(f"Sum without collecting outputs: {final}")
print()

# Example 3: Computing squares
xs = jnp.arange(5)

def square_body(carry, x):
    return carry + 1, x ** 2  # count iterations, output squares

final_count, squares = jax.lax.scan(square_body, 0, xs)
print(f"Final count: {final_count}")
print(f"Squares:     {squares}")
print()

# Example 4: Iterating over a 2D array (iterate over rows)
matrix = jnp.array([[1, 2, 3], 
                    [4, 5, 6], 
                    [7, 8, 9]])

def row_sum_body(carry, row):
    return carry + row.sum(), row.sum()

final, row_sums = jax.lax.scan(row_sum_body, jnp.array(0.0), matrix)
print(f"Matrix:\n{matrix}")
print(f"Row sums:     {row_sums}")
print(f"Total sum:    {final}")

Input:       [1. 2. 3. 4. 5.]
Final sum:   15.0
All running: [ 1.  3.  6. 10. 15.]

Sum without collecting outputs: 15.0

Final count: 5
Squares:     [ 0  1  4  9 16]

Matrix:
[[1 2 3]
 [4 5 6]
 [7 8 9]]
Row sums:     [ 6 15 24]
Total sum:    45.0


# The carry concept

The **carry** is the state passed between iterations. It's how loops "remember" 
things from previous steps.

## What is the carry?

- A value (or pytree of values) that represents the loop's state
- Updated by the body function each iteration
- Passed automatically from one iteration to the next

## Examples of carry

| Carry type | Use case |
|-----------|----------|
| Running sum | Cumulative sum of inputs |
| Running max | Maximum seen so far |
| Counter | Number of iterations completed |
| List of tokens | Generated sequence so far |
| KV cache | Cached attention keys/values |
| Pytree of params | Accumulator over model parameters |

## The carry can be ANY pytree

You can carry:
- A scalar: `jnp.array(0.0)`
- A 1D array: `jnp.zeros(10)`
- A dict: `{"sum": 0.0, "count": 0}`
- A complex pytree: `{"layer1": {"k": ..., "v": ...}, "layer2": ...}`

## Mental model

```
Iteration 0: carry = init,         x = xs[0]  → new_carry_0
Iteration 1: carry = new_carry_0,  x = xs[1]  → new_carry_1
Iteration 2: carry = new_carry_1,  x = xs[2]  → new_carry_2
...
Final:       final = new_carry_N-1
```

The body function is purely functional: same inputs → same outputs. 
This is what makes JAX's transformations (jit, grad, vmap) work.

In [3]:
# Example 1: Running statistics (mean and variance using Welford's algorithm)
xs = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])

def stats_body(carry, x):
    mean, count, m2 = carry  # carry is a TUPLE of 3 values
    new_count = count + 1
    delta = x - mean
    new_mean = mean + delta / new_count
    delta2 = x - new_mean
    new_m2 = m2 + delta * delta2
    return (new_mean, new_count, new_m2), new_mean

init = (jnp.array(0.0), 0, jnp.array(0.0))  # (mean, count, M2)
final, running_means = jax.lax.scan(stats_body, init, xs)

final_mean, final_count, final_m2 = final
final_variance = final_m2 / final_count

print(f"Final mean:     {final_mean}")
print(f"Final variance: {final_variance}")
print(f"Running means:  {running_means}")
print()

# Example 2: Carry as a DICT
def counter_body(carry, x):
    # carry = {"count": ..., "sum": ..., "max": ...}
    new_carry = {
        "count": carry["count"] + 1,
        "sum": carry["sum"] + x,
        "max": jnp.maximum(carry["max"], x),
    }
    return new_carry, new_carry

init = {"count": 0, "sum": jnp.array(0.0), "max": jnp.array(-float("inf"))}
final, all_states = jax.lax.scan(counter_body, init, xs)

print(f"Final state: {final}")
print(f"All counts:  {all_states['count']}")
print(f"All maxes:   {all_states['max']}")

Final mean:     5.5
Final variance: 8.25
Running means:  [1.  1.5 2.  2.5 3.  3.5 4.  4.5 5.  5.5]

Final state: {'count': Array(10, dtype=int32, weak_type=True), 'max': Array(10., dtype=float32), 'sum': Array(55., dtype=float32)}
All counts:  [ 1  2  3  4  5  6  7  8  9 10]
All maxes:   [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10.]


# Unrolling and compilation

When you use `lax.scan` with `jax.jit`, JAX compiles the entire loop into 
a single optimized program.

## What is unrolling?

"Loop unrolling" means: instead of executing the loop repeatedly at runtime, 
the compiler generates explicit code for each iteration upfront.

**Without unrolling (Python loop):**
```python
for i in range(1000):
    body(state, xs[i])  # 1000 separate operations
```

**With unrolling (scan + JIT):**
```
body(state, xs[0])
body(result_0, xs[1])
body(result_1, xs[2])
...
```
All pre-compiled, all fused into optimized GPU kernels.

## Trade-offs

| Aspect | Python loop | lax.scan + jit |
|--------|------------|----------------|
| Speed | Slow (per-iter overhead) | Fast (compiled) |
| Memory | Low (one step at a time) | Higher (may unroll all iters) |
| Compilation | None | Slow first time, fast after |
| Flexibility | Any control flow | Static structure |

## The `unroll` parameter

`lax.scan` has an `unroll` parameter that controls how many iterations 
are unrolled at a time:
- `unroll=1`: Don't unroll (lowest memory, but less optimization)
- `unroll=N`: Unroll N iterations at a time (faster, more memory)
- Default: JAX picks based on the loop length

For decode loops (100-2000 tokens), default is usually fine.

In [7]:
# A more complex body to demonstrate compilation benefits
def body(carry, x):
    a, b, c = carry
    new_a = jnp.sin(a) + jnp.cos(x)
    new_b = b * 0.99 + x * 0.01
    new_c = jnp.maximum(c, x)
    return (new_a, new_b, new_c), new_a

xs = jnp.arange(500)

# Python loop version
def python_loop(xs):
    carry = (jnp.array(0.0), jnp.array(0.0), jnp.array(-1e9))
    outputs = []
    for x in xs:
        carry, out = body(carry, x)
        outputs.append(out)
    return carry, jnp.stack(outputs)

# Scan version
def scan_version(xs):
    init = (jnp.array(0.0), jnp.array(0.0), jnp.array(-1e9))
    return jax.lax.scan(body, init, xs)

# JIT compile scan version
scan_jit = jax.jit(scan_version)

# Warmup
_ = python_loop(xs)
_ = scan_jit(xs)[1].block_until_ready()

# Time both
N = 50

start = time.time()
for _ in range(N):
    python_loop(xs)[1].block_until_ready()
t_python = time.time() - start

start = time.time()
for _ in range(N):
    scan_jit(xs)[1].block_until_ready()
t_scan = time.time() - start

print(f"Python loop: {t_python*1000:.2f} ms (avg of {N} runs)")
print(f"scan + jit:  {t_scan*1000:.4f} ms (avg of {N} runs)")
print(f"Speedup:     {t_python/t_scan:.0f}x")
print()

# Compare with unroll
scan_unroll = jax.jit(scan_version, static_argnames=())  # default unroll
scan_unrolled = jax.jit(lambda x: jax.lax.scan(body, 
    (jnp.array(0.0), jnp.array(0.0), jnp.array(-1e9)), x, unroll=10))

_ = scan_unrolled(xs)[1].block_until_ready()

start = time.time()
for _ in range(N):
    scan_unrolled(xs)[1].block_until_ready()
t_unrolled = time.time() - start

print(f"scan + jit (unroll=10): {t_unrolled*1000:.4f} ms")

Python loop: 2634.94 ms (avg of 50 runs)
scan + jit:  1.1282 ms (avg of 50 runs)
Speedup:     2336x

scan + jit (unroll=10): 1.2054 ms


# lax.while_loop

`lax.while_loop` is for loops where you **don't know the count in advance**.

## When to use while_loop instead of scan

- **scan**: Iterate over a fixed-length array
- **while_loop**: Iterate until a condition is met (e.g., generate until EOS token)

## LLM decode use case

In autoregressive text generation:
- You don't know how many tokens the model will produce
- You generate until the model outputs an EOS (end-of-sequence) token
- This is the PERFECT use case for `lax.while_loop`

## Function signature

```python
jax.lax.while_loop(cond_fn, body_fn, init)
```

- `cond_fn(carry) -> bool`: Returns whether to continue
- `body_fn(carry) -> new_carry`: One iteration of the loop
- `init`: Initial carry

## Important constraint

The condition must be **jit-compatible**. You can't use Python control flow 
in `cond_fn`. The condition must be a traced JAX boolean.

Example: `jnp.logical_not(jnp.all(done))` works. 
Python `if done: ...` doesn't.

## Pattern for decode

```python
def cond_fn(carry):
    tokens, done = carry
    return jnp.logical_not(jnp.all(done))

def body_fn(carry):
    tokens, done = carry
    new_token = model.generate_next(tokens)
    new_tokens = tokens.at[:, -1].set(new_token)
    new_done = jnp.logical_or(done, new_token == EOS_TOKEN)
    return (new_tokens, new_done)

final, _ = jax.lax.while_loop(cond_fn, body_fn, init_carry)
```

In [5]:
# Example 1: Generate random numbers until you hit one > 0.95
def cond_fn(carry):
    x, count = carry
    return jnp.logical_and(x < 0.95, count < 100)  # safety limit

def body_fn(carry):
    x, count = carry
    new_x = jax.random.uniform(jax.random.PRNGKey(count), ())
    return (new_x, count + 1)

init = (jnp.array(0.0), 0)
final_x, final_count = jax.lax.while_loop(cond_fn, body_fn, init)
print(f"Final x: {final_x:.4f}, iterations: {final_count}")
print()

# Example 2: Find first index where array value exceeds threshold
arr = jnp.array([0.1, 0.3, 0.2, 0.8, 0.9, 0.5, 0.4])

def find_cond(carry):
    idx, found, value = carry
    return jnp.logical_and(idx < len(arr), jnp.logical_not(found))

def find_body(carry):
    idx, found, value = carry
    new_value = arr[idx]
    new_found = jnp.logical_or(found, new_value > 0.7)
    return (idx + 1, new_found, new_value)

init = (0, False, jnp.array(0.0))
final_idx, final_found, final_value = jax.lax.while_loop(find_cond, find_body, init)
print(f"Found value > 0.7 at index {final_idx - 1}, value = {final_value}")
print()

# Example 3: Simulating decode until EOS
# EOS token = 2, vocab size = 10, max length = 20
EOS = 2
MAX_LEN = 20

def decode_cond(carry):
    tokens, step = carry
    not_done = jnp.logical_and(
        jnp.logical_not(jnp.any(tokens == EOS)),
        step < MAX_LEN
    )
    return not_done

def decode_body(carry):
    tokens, step = carry
    # Simulate generating a token (in real code, this is model forward pass)
    key = jax.random.PRNGKey(step)
    new_token = jax.random.randint(key, (1,), 0, 10)
    new_tokens = tokens.at[0, step].set(new_token[0])
    return (new_tokens, step + 1)

init = (jnp.zeros((1, MAX_LEN), dtype=jnp.int32), 0)
final_tokens, final_step = jax.lax.while_loop(decode_cond, decode_body, init)
print(f"Final tokens: {final_tokens}")
print(f"Final step: {final_step}")
print(f"Stop reason: {'EOS' if jnp.any(final_tokens == EOS) else 'max length'}")

Final x: 0.9976, iterations: 66

Found value > 0.7 at index 3, value = 0.800000011920929

Final tokens: [[9 6 9 5 4 1 2 0 0 0 0 0 0 0 0 0 0 0 0 0]]
Final step: 7
Stop reason: EOS


# Common scan patterns

These patterns appear constantly in JAX code. Learn them well.

## Pattern 1: Accumulator (reduce)

Reduce an array to a single value:

```python
def body(acc, x):
    return acc + x, None  # discard per-iter output

total, _ = jax.lax.scan(body, 0.0, xs)
```

## Pattern 2: Map (transform every element)

Apply a function to every element, collect outputs:

```python
def body(_, x):
    return None, f(x)  # carry is unused, output is the result

_, results = jax.lax.scan(body, None, xs)
```

## Pattern 3: Running statistics

Track running mean/variance/max/etc.:

```python
def body(state, x):
    mean, count = state
    new_count = count + 1
    new_mean = mean + (x - mean) / new_count
    return (new_mean, new_count), new_mean
```

## Pattern 4: Sequence generation

Generate a sequence step-by-step (like LLM decode):

```python
def body(tokens, _):
    new_token = model_step(tokens)  # forward pass
    new_tokens = tokens.at[:, -1].set(new_token)
    return new_tokens, new_token

final_tokens, generated = jax.lax.scan(body, init_tokens, None, length=max_len)
```

The trick: pass `xs=None` and use `length=N` to iterate N times.

## Pattern 5: Recurrence with memory

Like Fibonacci, where state depends on multiple previous values:

```python
def body(state, _):
    a, b = state
    return (b, a + b), a

# Generates Fibonacci numbers
```

## Pattern 6: Pairwise operations

Compare each element with the previous:

```python
def body(prev, x):
    return x, jnp.abs(x - prev)  # output difference from prev

# Note: prev is the PREVIOUS x, not x[i-1] in the array
```

In [8]:
xs = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])

# Pattern 1: Accumulator (sum)
def body(acc, x):
    return acc + x, None
total, _ = jax.lax.scan(body, 0.0, xs)
print(f"Sum: {total}")
print()

# Pattern 2: Map (square every element)
def body(_, x):
    return None, x ** 2
_, squares = jax.lax.scan(body, None, xs)
print(f"Squares: {squares}")
print()

# Pattern 3: Running mean
def body(state, x):
    mean, count = state
    new_count = count + 1
    new_mean = mean + (x - mean) / new_count
    return (new_mean, new_count), new_mean

(init_mean, init_count) = (jnp.array(0.0), 0)
final, running_means = jax.lax.scan(body, (init_mean, init_count), xs)
print(f"Running means: {running_means}")
print(f"Final mean: {final[0]} (expected: {xs.mean()})")
print()

# Pattern 4: Sequence generation (no xs, use length)
def generate_body(tokens, _):
    # Simulate: last token becomes the "next" token
    last_token = tokens[-1]
    new_token = (last_token + 1) % 10
    new_tokens = jnp.append(tokens[1:], new_token)
    return new_tokens, new_token

init_seq = jnp.zeros(3, dtype=jnp.int32)
final_seq, generated = jax.lax.scan(
    generate_body, init_seq, None, length=10
)
print(f"Generated sequence: {generated}")
print(f"Final sliding window: {final_seq}")
print()

# Pattern 5: Fibonacci
def fib_body(state, _):
    a, b = state
    return (b, a + b), a

init = (jnp.array(0), jnp.array(1))
final, fib_seq = jax.lax.scan(fib_body, init, None, length=10)
print(f"Fibonacci: {fib_seq}")

Sum: 55.0

Squares: [  1.   4.   9.  16.  25.  36.  49.  64.  81. 100.]

Running means: [1.  1.5 2.  2.5 3.  3.5 4.  4.5 5.  5.5]
Final mean: 5.5 (expected: 5.5)

Generated sequence: [1 2 3 4 5 6 7 8 9 0]
Final sliding window: [8 9 0]

Fibonacci: [ 0  1  1  2  3  5  8 13 21 34]


# Building toward the decode loop

This is where everything comes together. We'll build up the structure of an 
LLM decode loop step by step.

## What is decode?

After the model processes the prompt (prefill), it generates tokens one at a time:
1. Take the last token's logits
2. Sample/greedy-pick the next token
3. Append it to the sequence
4. Run the model again with the new token
5. Repeat until EOS or max length

## Why scan is perfect for this

Each step:
- Takes the current sequence (carry)
- Computes the next token (output)
- Updates the sequence (new carry)

```python
def decode_step(sequence, _):
    # Forward pass: model(sequence)
    logits = model(sequence)
    next_token = jnp.argmax(logits[:, -1, :], axis=-1)
    # Append next_token to sequence
    new_sequence = jnp.append(sequence, next_token)
    return new_sequence, next_token

# Run for max_len iterations
final_seq, all_tokens = jax.lax.scan(
    decode_step, prompt, None, length=max_new_tokens
)
```

## Full decode with KV cache

In Notebook 05, you'll extend this to:
- Carry the KV cache as part of the state
- Pre-allocate the cache before generation
- Update only the new positions each iteration

## The shape of the decode loop

```
Iteration 0: model(prompt)         → logits → next_token_0
Iteration 1: model(prompt + t_0)   → logits → next_token_1
Iteration 2: model(prompt + t_0:t_1) → logits → next_token_2
...
```

Each iteration uses the FULL sequence so far. With KV cache, each iteration 
only processes ONE new token (much faster).

## For now: understand the structure

In this notebook, we're just building the scan-based loop. The actual model 
forward pass comes in Notebook 04. The KV cache comes in Notebook 05.

In [11]:
# Simulate a tiny "model": given a sequence, the next token is the sum mod 10
def fake_model(sequence):
    """Given current sequence, predict next token (sum mod 10)."""
    valid_part = sequence[jnp.argmax(sequence > 0)]  # ignore zeros
    next_val = jnp.sum(sequence) % 10
    logits = jnp.zeros(10).at[next_val].set(1.0)
    return logits

def greedy(logits):
    return jnp.argmax(logits, axis=-1)

# Pre-allocate the FULL sequence (prompt + max new tokens)
prompt = jnp.array([1, 2, 3, 4])
max_new_tokens = 5
total_len = len(prompt) + max_new_tokens  # 9

# Allocate and fill with prompt
sequence = jnp.zeros(total_len, dtype=jnp.int32)
sequence = sequence.at[:len(prompt)].set(prompt)

def decode_step(carry, step_idx):
    sequence, position = carry
    
    # Get next token (in real code, this is the model forward pass)
    logits = fake_model(sequence[:position])
    next_token = greedy(logits)
    
    # Write next_token at the current position
    new_sequence = sequence.at[position].set(next_token)
    
    return (new_sequence, position + 1), next_token

# Run scan: iterate max_new_tokens times
init = (sequence, len(prompt))
(final_seq, final_pos), generated = jax.lax.scan(
    decode_step, init, jnp.arange(max_new_tokens)
)

print(f"Initial prompt:    {prompt}")
print(f"Generated tokens:  {generated}")
print(f"Final sequence:    {final_seq}")
print(f"Final position:    {final_pos}")

# Verification
print("\nVerification:")
seq = list(prompt)
for i, t in enumerate(generated):
    expected = sum(seq) % 10
    match = int(t) == expected
    print(f"  Step {i}: seq so far={seq}, expected={expected}, got={int(t)}, match={match}")
    seq.append(int(t))

IndexError: Array slice indices must have static start/stop/step to be used with NumPy indexing syntax. Found slice(None, JitTracer<~int32[]>, None). To index a statically sized array at a dynamic position, try lax.dynamic_slice/dynamic_update_slice (JAX does not support dynamically sized arrays within JIT compiled functions).

In [10]:
# Simulate a tiny "model": given a sequence, the next token is the sum mod 10
def fake_model(sequence):
    """Returns logits (probabilities would come from softmax in real model)."""
    next_val = jnp.sum(sequence) % 10
    logits = jnp.zeros(10)
    logits = logits.at[next_val].set(1.0)
    return logits

# Greedy decode: pick the argmax
def greedy(logits):
    return jnp.argmax(logits, axis=-1)

# Single decode step
def decode_step(sequence, _):
    logits = fake_model(sequence)
    next_token = greedy(logits)
    new_sequence = jnp.append(sequence, next_token)
    return new_sequence, next_token

# Initial prompt
prompt = jnp.array([1, 2, 3, 4])
max_new_tokens = 5

# Run decode loop
final_sequence, generated = jax.lax.scan(decode_step, prompt, None, length=max_new_tokens)

print(f"Initial prompt:  {prompt}")
print(f"Generated tokens: {generated}")
print(f"Final sequence:  {final_sequence}")
print()

# Verify: each new token should equal sum of sequence so far, mod 10
print("Verification:")
seq = list(prompt)
for i, t in enumerate(generated):
    expected = sum(seq) % 10
    print(f"  Step {i}: seq={seq}, sum%10={expected}, generated={t}, match={t == expected}")
    seq.append(int(t))

TypeError: scan body function carry input and carry output must have equal types, but they differ:

The input carry sequence has type int32[4] but the corresponding output carry component has type int32[5], so the shapes do not match.

Revise the function so that all output types match the corresponding input types.